# WAF Sensor Analysis Toolkit
## Analyze Obfuscated Bot-Detection Scripts (Imperva/Incapsula)

This notebook provides tools to:
- Detect bot-detection and anti-automation mechanisms
- Deobfuscate JavaScript strings and functions
- Identify RC4 encryption patterns
- Generate bypass strategies

**Educational use only** - For authorized security testing, CTF challenges, and learning.

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q requests beautifulsoup4 lxml

## Step 2: Load Analysis Tools

In [ ]:
import re
import base64
import json
from typing import Dict, List, Tuple, Any

class JSDeobfuscator:
    """Analyzes and deobfuscates obfuscated JavaScript."""

    def __init__(self, code: str):
        self.code = code
        self.decoded_strings: Dict[int, str] = {}
        self.suspicious_patterns = []

    def decode_hex_escape_sequences(self) -> Dict[int, str]:
        """Extract and decode hex escape sequences (\\x**)."""
        hex_pattern = r"\\x([0-9a-fA-F]{2})"
        matches = re.findall(hex_pattern, self.code)

        decoded = {}
        for i, hex_byte in enumerate(matches):
            char = chr(int(hex_byte, 16))
            decoded[i] = char

        self.decoded_strings = decoded
        return decoded

    def extract_string_arrays(self) -> List[str]:
        """Extract and decode string arrays."""
        array_pattern = r"var\s+_0x[a-f0-9]+\s*=\s*\[(.*?)\]"
        matches = re.findall(array_pattern, self.code, re.DOTALL)

        decoded_arrays = []
        for array_content in matches:
            strings = re.findall(r"'([^']*)'", array_content)
            decoded_strs = []

            for s in strings:
                decoded = re.sub(
                    r"\\x([0-9a-fA-F]{2})",
                    lambda m: chr(int(m.group(1), 16)),
                    s,
                )
                decoded_strs.append(decoded)

            if decoded_strs:
                decoded_arrays.append("\n".join(decoded_strs))

        return decoded_arrays

    def detect_suspicious_patterns(self) -> List[Tuple[str, str]]:
        """Detect common WAF sensor evasion and anti-debugging patterns."""
        patterns = {
            "debugger": r"debugger\s*;",
            "console_disable": r"console\s*=\s*[{}]",
            "eval": r"\beval\s*\(",
            "Function": r"\bFunction\s*\(",
            "webdriver_check": r"navigator\.webdriver",
            "headless_detect": r"HeadlessChrome|PhantomJS|webdriver",
            "chrome_detect": r"chrome\.runtime|webextension",
            "proxy_detect": r"proxy|bypass|automation",
            "base64_decode": r"atob\s*\(",
            "base64_encode": r"btoa\s*\(",
            "rc4_ksa": r"KSA|RC4|ksa",
            "hex_encoding": r"\\x[0-9a-f]{2}",
        }

        suspicious = []
        for name, pattern in patterns.items():
            matches = re.findall(pattern, self.code, re.IGNORECASE)
            if matches:
                suspicious.append((name, f"Found {len(matches)} occurrence(s)"))

        return suspicious

    def analyze(self) -> Dict[str, Any]:
        """Run full analysis on the obfuscated code."""
        return {
            "total_length": len(self.code),
            "hex_sequences": len(self.decode_hex_escape_sequences()),
            "string_arrays": self.extract_string_arrays(),
            "suspicious_patterns": self.detect_suspicious_patterns(),
        }


class AntiDetectionAnalyzer:
    """Analyzes code for anti-detection and anti-automation patterns."""

    DETECTION_PATTERNS = {
        "headless_browser": {
            "patterns": [r"HeadlessChrome", r"navigator\.webdriver", r"PhantomJS"],
            "description": "Detects headless browser execution",
            "bypass": "Override User-Agent, patch navigator.webdriver",
        },
        "browser_automation": {
            "patterns": [r"webdriver", r"selenium", r"puppeteer", r"playwright"],
            "description": "Detects browser automation tools",
            "bypass": "Use stealth plugins, patch navigator properties",
        },
        "nodejs_detection": {
            "patterns": [r"process\.env", r"require\s*\(", r"__dirname"],
            "description": "Detects Node.js environment",
            "bypass": "Use browser environment (Playwright headless)",
        },
        "timing_attacks": {
            "patterns": [r"setTimeout", r"performance\.timing", r"Date\.now\(\)"],
            "description": "Uses timing analysis to detect automation",
            "bypass": "Override timing functions to return consistent values",
        },
        "console_disable": {
            "patterns": [r"console\s*=\s*\{\}", r"console\.log\s*="],
            "description": "Disables console for anti-debugging",
            "bypass": "Restore console before code runs",
        },
        "eval_detection": {
            "patterns": [r"eval\s*\(", r"Function\s*\("],
            "description": "Uses eval to dynamically load code",
            "bypass": "Analyze eval'd code separately or patch eval",
        },
    }

    def __init__(self, code: str):
        self.code = code
        self.findings: Dict[str, List[str]] = {}

    def analyze(self) -> Dict[str, List[str]]:
        """Run analysis on the code."""
        for check_name, check_info in self.DETECTION_PATTERNS.items():
            matches = []
            for pattern in check_info["patterns"]:
                found = re.findall(pattern, self.code, re.IGNORECASE)
                if found:
                    matches.extend(found)

            if matches:
                self.findings[check_name] = list(set(matches))

        return self.findings

    def get_bypass_strategies(self) -> Dict[str, str]:
        """Get bypass strategies for detected checks."""
        strategies = {}
        for check_name in self.findings:
            if check_name in self.DETECTION_PATTERNS:
                strategies[check_name] = (
                    self.DETECTION_PATTERNS[check_name]["bypass"]
                )
        return strategies

print("✓ Analysis tools loaded")

## Step 3: Define Analysis Functions

In [ ]:
def analyze_code(code: str):
    """Analyze JavaScript code for obfuscation and detection mechanisms."""
    
    print("\n" + "="*80)
    print("ANTI-DETECTION ANALYSIS")
    print("="*80)
    
    detector = AntiDetectionAnalyzer(code)
    findings = detector.analyze()
    strategies = detector.get_bypass_strategies()
    
    if not findings:
        print("\n✓ No obvious anti-detection patterns found")
    else:
        print(f"\n⚠️  Found {len(findings)} detection mechanism(s):\n")
        for i, (check_name, patterns) in enumerate(findings.items(), 1):
            print(f"{i}. {check_name.upper().replace('_', ' ')}")
            print(f"   Patterns: {', '.join(patterns[:2])}")
            if len(patterns) > 2:
                print(f"             (+ {len(patterns)-2} more)")
            print(f"   Bypass:   {strategies.get(check_name, 'N/A')}\n")
    
    print("\n" + "="*80)
    print("OBFUSCATION ANALYSIS")
    print("="*80 + "\n")
    
    deobf = JSDeobfuscator(code)
    results = deobf.analyze()
    
    print(f"File size: {results['total_length']} bytes")
    print(f"Hex-encoded strings: {results['hex_sequences']}")
    print(f"String arrays detected: {len(results['string_arrays'])}")
    
    if results['string_arrays']:
        print("\n📋 Decoded Strings (first array):")
        first_array = results['string_arrays'][0].split('\n')[:15]
        for i, s in enumerate(first_array, 1):
            print(f"  {i}. {s[:70]}")
        if len(results['string_arrays'][0].split('\n')) > 15:
            print(f"  ... ({len(results['string_arrays'][0].split(chr(10))) - 15} more)")
    
    print(f"\n🚨 Suspicious patterns:")
    for pattern, desc in results['suspicious_patterns']:
        print(f"  - {pattern}: {desc}")
    
    return {
        'detections': findings,
        'obfuscation': results
    }

def analyze_sensor_url(url: str):
    """Download and analyze a sensor script from URL."""
    import requests
    
    print(f"\n📥 Downloading from {url}...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        code = response.text
        print(f"✓ Downloaded {len(code)} bytes\n")
        return analyze_code(code)
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✓ Analysis functions defined")

---

# Usage Examples

Choose one of the options below to start analyzing WAF sensors:

## Option A: Analyze Sample Code

In [ ]:
# Sample obfuscated JavaScript (Imperva-like)
sample_code = r'''
var _0x6911 = [
    '\x63\x6f\x6f\x6b\x69\x65',  // 'cookie'
    '\x74\x6f\x6b\x65\x6e',      // 'token'
    '\x76\x65\x72\x69\x66\x79',  // 'verify'
    '\x6e\x61\x76\x69\x67\x61\x74\x6f\x72',  // 'navigator'
    '\x77\x65\x62\x64\x72\x69\x76\x65\x72',  // 'webdriver'
    '\x48\x65\x61\x64\x6c\x65\x73\x73\x43\x68\x72\x6f\x6d\x65'  // 'HeadlessChrome'
];

function _0x1691(a) { return _0x6911[a]; }

if (navigator.webdriver) { console.log('detected'); }
console.log = function() {};
eval(atob('base64payload'));
'''

results = analyze_code(sample_code)

## Option B: Download from URL

In [ ]:
# Analyze a real Incapsula sensor (example URL)
# Replace with your actual sensor URL
sensor_url = "https://events-new.mepha.ch/_Incapsula_Resource?SWJIYLWA=719d34d31c8e3a6e6fffd425f7e032f3"

# results = analyze_sensor_url(sensor_url)
print("To analyze a URL, uncomment the line above and replace with your sensor URL")

## Option C: Upload Your Own File

In [ ]:
from google.colab import files

print("Click the 'Choose Files' button below to upload a JavaScript file:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\nAnalyzing {filename}...")
    with open(filename, 'r', encoding='utf-8', errors='replace') as f:
        code = f.read()
    results = analyze_code(code)

---

## Export Results

In [ ]:
# Save results as JSON
if 'results' in locals() and results:
    import json
    with open('analysis_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print("✓ Results saved to analysis_results.json")
    
    # Download the file
    files.download('analysis_results.json')
else:
    print("No results to export. Run one of the analysis cells first.")

---

## Quick Reference: Detected Mechanisms

### Headless Browser Detection
```javascript
navigator.webdriver  // Automated browser marker
HeadlessChrome       // User-Agent string
PhantomJS            // Older headless browser
```
**Bypass:** Override User-Agent, patch `navigator.webdriver`

### Browser Automation Detection
```javascript
selenium, puppeteer, playwright  // Known automation tools
webdriver                        // W3C WebDriver standard
```
**Bypass:** Use stealth mode plugins, patch navigator properties

### Console Disabling (Anti-Debugging)
```javascript
console.log = function() {}
console = {}  // Full disable
```
**Bypass:** Restore console before code execution

### RC4 Encryption Indicators
```javascript
KSA, PRGA      // RC4 algorithm components
charCodeAt()   // Extract key bytes
atob()         // Base64 decode encrypted payload
```
**Bypass:** Identify RC4 key, decrypt payload

### Eval-Based Code Execution
```javascript
eval()           // Direct eval
Function()       // Constructor eval
indirect eval    // Using temporary variables
```
**Bypass:** Analyze eval'd code separately, patch if possible

---

## Next Steps

### 1. Implement Browser-Based Bypass (Playwright)
```python
from playwright.async_api import async_playwright

async def bypass_with_playwright():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        
        # Apply stealth patches
        await context.add_init_script('''
            Object.defineProperty(navigator, "webdriver", 
                {get: () => undefined});
        ''')
        
        page = await context.new_page()
        await page.goto('https://target.com/')
        content = await page.content()
        await browser.close()
        return content
```

### 2. Implement Algorithm-Based Bypass (Pure Python)
```python
# Extract algorithm from deobfuscated strings
# Reverse-engineer token computation
# Forge cookies without browser

def compute_token(nonce):
    # From analysis, algorithm is revealed
    return reverse(nonce) + hex_encode(sha256(nonce))

cookies = {'incap_ses_': compute_token(nonce)}
response = requests.get(url, cookies=cookies)
```

### 3. Extract and Try RC4 Keys
- Look for hard-coded keys in deobfuscated strings
- Try common keys: `imperva`, `cloudflare`, `sensor_key`, `bot_manager`
- Extract key from HTML comments or HTTP headers
- Use dynamic analysis to capture key at runtime